# 08. Sliding Window Merge (슬라이딩 윈도우 + 통합) - 리드타임 적용

Step 2에서 생성한 1시간 간격의 ```observation_end``` (예측 기준점)를 활용해 모델이 판단을 내리는 '현재 시점'을 정의하고, 

Step 4에서 해당 시점으로부터 설정된 리드타임(Lead Time) 이후에 발생하는 이벤트만을 긍정 레이블로 정의함으로써, 의료지의 조기 대응 시간을 보장하는 실무적인 예측 환경을 구축

## 목적
1. 코호트 기준 슬라이딩 윈도우 생성
2. 모든 raw 테이블을 윈도우 기준으로 집계 & 병합
3. 레이블 생성 (death, vent, pressor)- 리트타임 적용

### 리드타임의 적용

1. 논리의 일관성:

모든 이벤트(사망, 기계환기, 승압제)에 동일한 1시간의 리드타임을 적용하여 의료진이 공통적으로 대응할 수 있는 최소 시간을 확보

2. 변수 관리 :

리드타임이 포함된 정답(라벨)을 기존 변수명에 덮어씌우거나 명확히 통일함으로써 향후 리드타임이 반영된 학습이 가능하도록 최적화


## 입력 (Raw 테이블들)
- `cohort_base.csv`: 기본 코호트
- `vital_raw.csv`: 활력징후
- `lab_raw.csv`: 검사 결과
- `ventilation_raw.csv`: 인공호흡기
- `pressor_raw.csv`: 승압제
- `urine_raw.csv`: 소변량
- `gcs_raw.csv`: GCS 점수

## 출력
- `sliding_window_merged.csv`: 슬라이딩 윈도우 통합 데이터

## 윈도우 설정
- **Window Size**: 6시간
- **Stride**: 1시간
- **Range**: ICU 입실 후 6h ~ 72h
- **예상 결과**: ~93만 rows (약 23K 환자 × 약 40 시점)

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta

# 설정
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

# 윈도우 파라미터
WINDOW_SIZE_H = 6   # 윈도우 크기 (시간)
STRIDE_H = 1        # 이동 간격 (시간)
MIN_HOUR = 6        # 시작 시점 (ICU 입실 후)
MAX_HOUR = 72       # 종료 시점

print("=== 10. Sliding Window Merge 시작 ===")
print(f"\n윈도우 설정:")
print(f"  - Window Size: {WINDOW_SIZE_H}h")
print(f"  - Stride: {STRIDE_H}h")
print(f"  - Range: {MIN_HOUR}h ~ {MAX_HOUR}h")

=== 10. Sliding Window Merge 시작 ===

윈도우 설정:
  - Window Size: 6h
  - Stride: 1h
  - Range: 6h ~ 72h


## Step 1: 코호트 및 Raw 테이블 로드

In [2]:
print("\nStep 1: 데이터 로드")

# 코호트
df_cohort = pd.read_csv(
    os.path.join(INPUT_DIR, 'cohort_base.csv'),
    parse_dates=['intime', 'outtime', 'deathtime', 'dnr_time', 'vent_start_time', 'pressor_start_time']
)
print(f"✓ cohort_base: {len(df_cohort):,}명")

# Raw 테이블들
df_vital = pd.read_csv(os.path.join(INPUT_DIR, 'vital_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ vital_raw: {len(df_vital):,} rows")

df_lab = pd.read_csv(os.path.join(INPUT_DIR, 'lab_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ lab_raw: {len(df_lab):,} rows")

df_vent = pd.read_csv(os.path.join(INPUT_DIR, 'ventilation_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ ventilation_raw: {len(df_vent):,} rows")

df_pressor = pd.read_csv(os.path.join(INPUT_DIR, 'pressor_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ pressor_raw: {len(df_pressor):,} rows")

df_urine = pd.read_csv(os.path.join(INPUT_DIR, 'urine_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ urine_raw: {len(df_urine):,} rows")

df_gcs = pd.read_csv(os.path.join(INPUT_DIR, 'gcs_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ gcs_raw: {len(df_gcs):,} rows")


Step 1: 데이터 로드
✓ cohort_base: 54,551명
✓ vital_raw: 5,337,227 rows
✓ lab_raw: 739,339 rows
✓ ventilation_raw: 1,859,863 rows
✓ pressor_raw: 639,618 rows
✓ urine_raw: 2,789,391 rows
✓ gcs_raw: 1,539,690 rows


## Step 2: 슬라이딩 윈도우 생성

In [3]:
print("\nStep 2: 슬라이딩 윈도우 생성")

# 각 환자별 윈도우 생성
windows = []

for _, row in df_cohort.iterrows():
    stay_id = row['stay_id']
    intime = row['intime']
    outtime = row['outtime']
    
    # 6h부터 72h까지 1시간 간격
    for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
        obs_end = intime + timedelta(hours=obs_hour)
        obs_start = intime + timedelta(hours=obs_hour - WINDOW_SIZE_H)
        
        # ICU 체류 중인 경우만
        if obs_end <= outtime:
            windows.append({
                'stay_id': stay_id,
                'observation_hour': obs_hour,
                'observation_start': obs_start,
                'observation_end': obs_end
            })

df_windows = pd.DataFrame(windows)
print(f"✓ 초기 윈도우 생성: {len(df_windows):,}개")

# 코호트 정보 병합
df_windows = df_windows.merge(
    df_cohort[['stay_id', 'subject_id', 'hadm_id', 'anchor_age', 'gender', 
               'first_careunit', 'deathtime', 'dnr_time', 'vent_start_time', 
               'pressor_start_time', 'icu_mortality', 'hospital_mortality']],
    on='stay_id',
    how='left'
)
print(f"✓ 코호트 정보 병합 완료")


Step 2: 슬라이딩 윈도우 생성
✓ 초기 윈도우 생성: 2,673,378개
✓ 코호트 정보 병합 완료


## Step 3: DNR/Event Censoring 적용

In [4]:
print("\nStep 3: DNR/Event Censoring 적용")

before_filter = len(df_windows)

# DNR 이전 윈도우만
df_windows = df_windows[
    (df_windows['dnr_time'].isna()) | 
    (df_windows['observation_end'] < df_windows['dnr_time'])
]

# 사망 이전 윈도우만
df_windows = df_windows[
    (df_windows['deathtime'].isna()) | 
    (df_windows['observation_end'] < df_windows['deathtime'])
]

# Ventilation 시작 이전 윈도우만
df_windows = df_windows[
    (df_windows['vent_start_time'].isna()) | 
    (df_windows['observation_end'] < df_windows['vent_start_time'])
]

# Pressor 시작 이전 윈도우만
df_windows = df_windows[
    (df_windows['pressor_start_time'].isna()) | 
    (df_windows['observation_end'] < df_windows['pressor_start_time'])
]

after_filter = len(df_windows)
print(f"✓ Censoring 적용: {before_filter:,} → {after_filter:,} ({after_filter/before_filter*100:.1f}%)")


Step 3: DNR/Event Censoring 적용
✓ Censoring 적용: 2,673,378 → 941,817 (35.2%)


## Step 4: 레이블 생성 (리드타임 적용 버전)

In [ ]:
print("\nStep 4: 레이블 생성")

# 리드타임 설정
LEAD_TIME_H = 1  # 1시간의 리드타임 설정
lead_delta = timedelta(hours=LEAD_TIME_H)

for horizon in [6, 12, 24]:
    horizon_delta = timedelta(hours=horizon)
    
    # 1. Death 레이블 (리드타임 적용)
    df_windows[f'death_next_{horizon}h'] = (
        (df_windows['deathtime'].notna()) &
        (df_windows['deathtime'] > df_windows['observation_end'] + lead_delta) & 
        (df_windows['deathtime'] <= df_windows['observation_end'] + lead_delta + horizon_delta)
    ).astype(int)
    
    # 2. Ventilation 레이블 (리드타임 적용)
    df_windows[f'vent_next_{horizon}h'] = (
        (df_windows['vent_start_time'].notna()) &
        (df_windows['vent_start_time'] > df_windows['observation_end'] + lead_delta) &
        (df_windows['vent_start_time'] <= df_windows['observation_end'] + lead_delta + horizon_delta)
    ).astype(int)
    
    # 3. Pressor 레이블 (리드타임 적용)
    df_windows[f'pressor_next_{horizon}h'] = (
        (df_windows['pressor_start_time'].notna()) &
        (df_windows['pressor_start_time'] > df_windows['observation_end'] + lead_delta) &
        (df_windows['pressor_start_time'] <= df_windows['observation_end'] + lead_delta + horizon_delta)
    ).astype(int)
    
    # 4. Composite (위의 세 가지 중 하나라도 발생하면 1)
    df_windows[f'composite_next_{horizon}h'] = (
        (df_windows[f'death_next_{horizon}h'] == 1) |
        (df_windows[f'vent_next_{horizon}h'] == 1) |
        (df_windows[f'pressor_next_{horizon}h'] == 1)
    ).astype(int)

print(f"\n=== 레이블 분포 (Lead Time: {LEAD_TIME_H}h) ===")
for horizon in [6, 12, 24]:
    print(f"\n{horizon}h 예측 (실제 구간: {LEAD_TIME_H}h ~ {LEAD_TIME_H + horizon}h 뒤):")
    print(f"  - Death: {df_windows[f'death_next_{horizon}h'].sum():,} ({df_windows[f'death_next_{horizon}h'].mean()*100:.2f}%)")
    print(f"  - Vent: {df_windows[f'vent_next_{horizon}h'].sum():,} ({df_windows[f'vent_next_{horizon}h'].mean()*100:.2f}%)")
    print(f"  - Pressor: {df_windows[f'pressor_next_{horizon}h'].sum():,} ({df_windows[f'pressor_next_{horizon}h'].mean()*100:.2f}%)")
    print(f"  - Composite: {df_windows[f'composite_next_{horizon}h'].sum():,} ({df_windows[f'composite_next_{horizon}h'].mean()*100:.2f}%)")


Step 4: 레이블 생성

=== 레이블 분포 ===

6h 예측:
  - Death: 1,745 (0.19%)
  - Vent: 8,995 (0.96%)
  - Pressor: 4,527 (0.48%)
  - Composite: 13,097 (1.39%)

12h 예측:
  - Death: 3,791 (0.40%)
  - Vent: 15,473 (1.64%)
  - Pressor: 8,127 (0.86%)
  - Composite: 22,910 (2.43%)

24h 예측:
  - Death: 8,670 (0.92%)
  - Vent: 24,286 (2.58%)
  - Pressor: 13,289 (1.41%)
  - Composite: 37,804 (4.01%)


## Step 5: 피처 테이블 병합 (윈도우 기준 집계)

In [6]:
print("\nStep 5: 피처 테이블 병합")

def aggregate_to_window(df_raw, df_windows, feature_cols, agg_func='mean'):
    """
    Raw 데이터를 윈도우 기준으로 집계
    """
    results = []
    
    for _, window in df_windows.iterrows():
        stay_id = window['stay_id']
        obs_start = window['observation_start']
        obs_end = window['observation_end']
        
        # 해당 윈도우 내 데이터 필터링
        mask = (
            (df_raw['stay_id'] == stay_id) &
            (df_raw['charttime_h'] >= obs_start) &
            (df_raw['charttime_h'] <= obs_end)
        )
        window_data = df_raw.loc[mask, feature_cols]
        
        if len(window_data) > 0:
            if agg_func == 'mean':
                agg_values = window_data.mean()
            elif agg_func == 'max':
                agg_values = window_data.max()
            elif agg_func == 'min':
                agg_values = window_data.min()
            elif agg_func == 'last':
                agg_values = window_data.iloc[-1]
            results.append(agg_values.to_dict())
        else:
            results.append({col: np.nan for col in feature_cols})
    
    return pd.DataFrame(results)

# 더 효율적인 방식: merge_asof 또는 groupby 활용
# 여기서는 간단한 버전으로 구현

print("  피처 집계 중... (시간이 걸릴 수 있음)")
print("  → 실제 운영 시에는 SQL 또는 벡터화된 방식 권장")


Step 5: 피처 테이블 병합
  피처 집계 중... (시간이 걸릴 수 있음)
  → 실제 운영 시에는 SQL 또는 벡터화된 방식 권장


In [7]:
# 효율적인 방식: 각 raw 테이블에 observation_hour 계산 후 병합
print("\n  효율적 병합 방식 사용...")

# Vital: charttime_h 기준으로 intime과의 차이 계산
df_vital_merged = df_vital.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_vital_merged['hours_since_admit'] = (
    (df_vital_merged['charttime_h'] - df_vital_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

# 6시간 윈도우 내 평균 계산
vital_cols = ['hr', 'rr', 'spo2', 'temp', 'sbp', 'dbp', 'mbp']

# 각 observation_hour에 대해 (obs_hour - 6) ~ obs_hour 범위의 평균
vital_agg = []
for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
    window_start = obs_hour - WINDOW_SIZE_H
    window_end = obs_hour
    
    mask = (
        (df_vital_merged['hours_since_admit'] > window_start) &
        (df_vital_merged['hours_since_admit'] <= window_end)
    )
    
    agg = df_vital_merged[mask].groupby('stay_id')[vital_cols].mean().reset_index()
    agg['observation_hour'] = obs_hour
    vital_agg.append(agg)

df_vital_agg = pd.concat(vital_agg, ignore_index=True)
print(f"✓ Vital 집계 완료: {len(df_vital_agg):,} rows")

# 윈도우와 병합
df_windows = df_windows.merge(
    df_vital_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ Vital 병합 완료")


  효율적 병합 방식 사용...
✓ Vital 집계 완료: 2,793,081 rows
✓ Vital 병합 완료


In [8]:
# Lab 동일한 방식으로 처리
df_lab_merged = df_lab.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_lab_merged['hours_since_admit'] = (
    (df_lab_merged['charttime_h'] - df_lab_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

lab_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']

lab_agg = []
for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
    # Lab은 더 긴 윈도우 사용 가능 (검사가 덜 빈번함)
    window_start = obs_hour - 24  # 24시간 이내 검사 사용
    window_end = obs_hour
    
    mask = (
        (df_lab_merged['hours_since_admit'] > window_start) &
        (df_lab_merged['hours_since_admit'] <= window_end)
    )
    
    # 최신값 사용
    agg = df_lab_merged[mask].sort_values('hours_since_admit').groupby('stay_id')[lab_cols].last().reset_index()
    agg['observation_hour'] = obs_hour
    lab_agg.append(agg)

df_lab_agg = pd.concat(lab_agg, ignore_index=True)
print(f"✓ Lab 집계 완료: {len(df_lab_agg):,} rows")

df_windows = df_windows.merge(
    df_lab_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ Lab 병합 완료")

✓ Lab 집계 완료: 2,940,162 rows
✓ Lab 병합 완료


In [9]:
# GCS 병합
df_gcs_merged = df_gcs.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_gcs_merged['hours_since_admit'] = (
    (df_gcs_merged['charttime_h'] - df_gcs_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

gcs_cols = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']

gcs_agg = []
for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
    window_start = obs_hour - WINDOW_SIZE_H
    window_end = obs_hour
    
    mask = (
        (df_gcs_merged['hours_since_admit'] > window_start) &
        (df_gcs_merged['hours_since_admit'] <= window_end)
    )
    
    # 최신값 사용
    agg = df_gcs_merged[mask].sort_values('hours_since_admit').groupby('stay_id')[gcs_cols].last().reset_index()
    agg['observation_hour'] = obs_hour
    gcs_agg.append(agg)

df_gcs_agg = pd.concat(gcs_agg, ignore_index=True)
print(f"✓ GCS 집계 완료: {len(df_gcs_agg):,} rows")

df_windows = df_windows.merge(
    df_gcs_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ GCS 병합 완료")

✓ GCS 집계 완료: 2,481,321 rows
✓ GCS 병합 완료


In [10]:
# Urine 병합
df_urine_merged = df_urine.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_urine_merged['hours_since_admit'] = (
    (df_urine_merged['charttime_h'] - df_urine_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

urine_cols = ['urine_ml', 'urine_ml_kg_hr', 'oliguria_flag']

urine_agg = []
for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
    window_start = obs_hour - WINDOW_SIZE_H
    window_end = obs_hour
    
    mask = (
        (df_urine_merged['hours_since_admit'] > window_start) &
        (df_urine_merged['hours_since_admit'] <= window_end)
    )
    
    # 합계와 평균
    agg = df_urine_merged[mask].groupby('stay_id').agg({
        'urine_ml': 'sum',
        'urine_ml_kg_hr': 'mean',
        'oliguria_flag': 'max'
    }).reset_index()
    agg.columns = ['stay_id', 'urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']
    agg['observation_hour'] = obs_hour
    urine_agg.append(agg)

df_urine_agg = pd.concat(urine_agg, ignore_index=True)
print(f"✓ Urine 집계 완료: {len(df_urine_agg):,} rows")

df_windows = df_windows.merge(
    df_urine_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ Urine 병합 완료")

✓ Urine 집계 완료: 2,358,927 rows
✓ Urine 병합 완료


## Step 6: 결과 확인 및 저장

In [11]:
print("\n" + "="*60)
print("최종 결과 요약")
print("="*60)

print(f"\n총 행 수: {len(df_windows):,}개")
print(f"고유 환자: {df_windows['stay_id'].nunique():,}명")
print(f"환자당 평균 윈도우: {len(df_windows) / df_windows['stay_id'].nunique():.1f}개")

print(f"\n=== 컬럼 목록 ({len(df_windows.columns)}개) ===")
for col in df_windows.columns:
    missing = df_windows[col].isna().mean() * 100
    print(f"  - {col}: {missing:.1f}% 결측")


최종 결과 요약

총 행 수: 941,817개
고유 환자: 23,938명
환자당 평균 윈도우: 39.3개

=== 컬럼 목록 (50개) ===
  - stay_id: 0.0% 결측
  - observation_hour: 0.0% 결측
  - observation_start: 0.0% 결측
  - observation_end: 0.0% 결측
  - subject_id: 0.0% 결측
  - hadm_id: 0.0% 결측
  - anchor_age: 0.0% 결측
  - gender: 0.0% 결측
  - first_careunit: 0.0% 결측
  - deathtime: 90.9% 결측
  - dnr_time: 88.6% 결측
  - vent_start_time: 94.8% 결측
  - pressor_start_time: 96.7% 결측
  - icu_mortality: 0.0% 결측
  - hospital_mortality: 0.0% 결측
  - death_next_6h: 0.0% 결측
  - vent_next_6h: 0.0% 결측
  - pressor_next_6h: 0.0% 결측
  - composite_next_6h: 0.0% 결측
  - death_next_12h: 0.0% 결측
  - vent_next_12h: 0.0% 결측
  - pressor_next_12h: 0.0% 결측
  - composite_next_12h: 0.0% 결측
  - death_next_24h: 0.0% 결측
  - vent_next_24h: 0.0% 결측
  - pressor_next_24h: 0.0% 결측
  - composite_next_24h: 0.0% 결측
  - hr: 0.9% 결측
  - rr: 2.1% 결측
  - spo2: 1.2% 결측
  - temp: 4.3% 결측
  - sbp: 2.5% 결측
  - dbp: 2.5% 결측
  - mbp: 2.7% 결측
  - sao2: 91.5% 결측
  - ph: 68.4% 결측
  - lactate: 71.6% 결

In [12]:
# 저장
output_path = os.path.join(OUTPUT_DIR, 'sliding_window_merged.csv')
df_windows.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: sliding_window_merged.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 경로: {output_path}")


✓ 저장 완료: sliding_window_merged.csv
  - 파일 크기: 271.55 MB
  - 경로: ../data/processed/sliding_window_merged.csv


In [13]:
print("\n=== 샘플 데이터 ===")
df_windows.head()


=== 샘플 데이터 ===


,stay_id,observation_hour,observation_start,observation_end,subject_id,hadm_id,anchor_age,gender,first_careunit,deathtime,...,platelets,potassium,sodium,gcs_eye,gcs_verbal,gcs_motor,gcs_total,urine_ml_6h,urine_ml_kg_hr_avg,oliguria_flag
0,30000831,6,2140-04-17 21:26:33,2140-04-18 03:26:33,15726459,22744101,78,M,Coronary Care Unit (CCU),NaT,...,311.0,3.8,141.0,3.0,1.0,4.0,8.0,675.0,2.848101,0.0
1,30000831,7,2140-04-17 22:26:33,2140-04-18 04:26:33,15726459,22744101,78,M,Coronary Care Unit (CCU),NaT,...,311.0,3.8,141.0,3.0,1.0,4.0,8.0,775.0,2.452532,0.0
2,30000831,8,2140-04-17 23:26:33,2140-04-18 05:26:33,15726459,22744101,78,M,Coronary Care Unit (CCU),NaT,...,285.0,3.8,144.0,3.0,1.0,4.0,8.0,650.0,2.742616,0.0
3,30000831,9,2140-04-18 00:26:33,2140-04-18 06:26:33,15726459,22744101,78,M,Coronary Care Unit (CCU),NaT,...,285.0,3.8,144.0,3.0,1.0,4.0,8.0,500.0,2.109705,0.0
4,30000831,10,2140-04-18 01:26:33,2140-04-18 07:26:33,15726459,22744101,78,M,Coronary Care Unit (CCU),NaT,...,285.0,3.8,144.0,3.0,1.0,4.0,8.0,500.0,2.109705,0.0


In [14]:
print("\n=== 08. Sliding Window Merge 완료 ===")


=== 08. Sliding Window Merge 완료 ===
